# Chapter 5 — Stochastic volatility and time-varying parameters

The executable counterpart of `docs/companion/06-ch5-mh-and-sv.md`: the SV model for UK inflation (example 4) in the handbook's mean-only form and in unobserved-components form, and the TVP-AR(1) with SV (example 5).

In [ ]:
from pathlib import Path
import os
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from macrotoolkit import api as mtk
from macrotoolkit.run import REPO_ROOT as REPO   # never walk parent directories (HANDOFF warning)

HB = REPO / "examples" / "handbook"
HEAVY = os.environ.get("HANDBOOK_HEAVY") == "1"

def fit(name):
    """Fit an example spec (reused by hash if the run exists) and print its record."""
    run = mtk.fit(str(HB / name / "spec.yaml"))
    mc = run.mirror_check
    print(f"{name}: run {run.hash} | verdict {run.verdict} | {'; '.join(run.diagnostics['reasons'])}")
    print(f"  fit-time mirror check: max |Stan - Python| KF loglik = {mc['max_abs_diff']:.2e} over {mc['n_points']} prior draws")
    return run

def heavy(name):
    if not HEAVY:
        print(f"{name}: heavy example -- run with HANDBOOK_HEAVY=1 (a run session); skipped here.")
        return None
    return fit(name)


## Example 4: the SV model in the handbook's own form — a constant mean, no state, the SV block on the shock

In [ ]:
run = fit("ch5_sv_uk_inflation")
run.param_table().round(4)

In [ ]:
run.outputs().figure("states")

## The same SV shock around a random-walk level (the form first run on the S7 branch)

In [ ]:
run = fit("ch5_sv_ucsv_form")
run.param_table().round(4)

In [ ]:
run.outputs().figure("states")

## Example 5: the TVP-AR(1) with SV — the handbook's four panels at three dates, and IRFs conditional on the coefficient state

In [ ]:
run = fit("ch5_tvp_ar1_sv")
run.param_table().round(4)

In [ ]:
sd = run.outputs().compute("states"); d = sd.dates
picks = [int(np.argmin(np.abs(d - np.datetime64(x)))) for x in ("1930-01-01", "1975-01-01", "2008-10-01")]
c, b = sd.states["c"], sd.states["b"]
print("b_t:", [round(float(np.median(b[:, i])), 2) for i in picks], "c_t:", [round(float(np.median(c[:, i])), 2) for i in picks],
      "long-run mean:", [round(float(np.median((c / (1 - b))[:, i])), 1) for i in picks], "exp(h/2):", [round(float(np.median(sd.vol["e"][:, i])), 2) for i in picks])
run.outputs().figure("states")

In [ ]:
irf = run.outputs().compute("irf")
print({k: round(float(np.median(v["e"]["pi"][:, 3])), 2) for k, v in irf.by_date.items()}, "| omitted:", list(irf.omitted_shocks))
run.outputs().figure("irf")

In [ ]:
plt.close("all")